In [1]:
from data import BasinDataLake

root_dir = "/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs/datalakes/training"
store = BasinDataLake(root_dir)

In [22]:
from deltalake import DeltaTable

table_uri = store._get_source_table_uri('gauge')
dt = DeltaTable(table_uri)
dt

DeltaTable()

In [5]:
df = store.read_dynamic(
    'ABOM-100288010', 
    # subbasin = None,
    # subbasin = 'ABOM-100288010', 
    source = 'gauge', 
    # start_date = '2020-01-01',
    # end_date = '2020-02-01',
)
df

gauge
Empty DataFrame
Columns: [date, discharge, sp_discharge, subbasin, year, basin]
Index: []


Empty DataFrame
Columns: [(gauge, discharge), (gauge, sp_discharge)]
Index: []

In [6]:
import pandas as pd
pq_df = pd.read_parquet('/nas/cee-water/cjgleason/ted/swot-ml/data/reservoirs/datalakes/training/dynamic_data/gauge/basin=ABOM-100288010/year=2020/part-00001-741b9a02-74da-477c-b5e8-db6cc2a820f4-c000.snappy.parquet')
pq_df[pq_df['subbasin'] == 'ABOM-100288010'].set_index('date').sort_index()

,discharge,sp_discharge,subbasin
date,,,
2020-01-01 00:00:00+00:00,0.310,0.000042,ABOM-100288010
2020-01-02 00:00:00+00:00,0.310,0.000042,ABOM-100288010
2020-01-03 00:00:00+00:00,0.318,0.000044,ABOM-100288010
2020-01-04 00:00:00+00:00,0.311,0.000043,ABOM-100288010
2020-01-05 00:00:00+00:00,0.308,0.000042,ABOM-100288010
...,...,...,...
2020-12-27 00:00:00+00:00,0.326,0.000045,ABOM-100288010
2020-12-28 00:00:00+00:00,0.316,0.000043,ABOM-100288010
2020-12-29 00:00:00+00:00,0.304,0.000042,ABOM-100288010


In [ ]:
pd.Timestamp('2020-01-01', tz='UTC').to_datetime64()

In [ ]:
pq_df['subbasin'].unique()

In [ ]:
from pathlib import Path
import xarray as xr
import numpy as np

zarr_path = Path('/scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2')
ds = xr.open_dataset(zarr_path / 'ABOM-138161010', engine='zarr')
ds

In [ ]:
ds['reach_q_b_river'].dtype

In [ ]:
for var in ds.data_vars:
    print(f"{var}: {np.isnan(ds[var].values).mean():0.3f}")

In [ ]:
# sources = self.list_sources()
sources

In [ ]:
self = store

basin_source_df = self.list_source_basin_data()
basin_source_map = {
    basin: list(row.index[row])
    for basin, row in basin_source_df.iterrows()
}
basin_source_map

In [ ]:
store.read_dynamic(
    basin='ABOM-100288010', 
    source=['era5','gauge'], 
    start_date='2024-01-01', 
    end_date='2024-12-31'
).droplevel(level='source', axis=1)

In [ ]:
import pandas as pd
from data import BasinDataLake

In [ ]:
store = BasinDataLake(root_dir="./my_delta_dataset")

In [ ]:
pwd

In [ ]:
# 2. Append data for multiple subbasins (this creates small files)
dates = pd.to_datetime(pd.date_range("2021-01-01", "2021-01-10", freq="D"))

source_dict = {
    'ERA5': pd.DataFrame(index=dates, data={'precip': range(10)}),
    'SWOT': pd.DataFrame(index=dates, data={'height': range(10)}),
    'S2': pd.DataFrame(index=dates, data={'width': range(10)}),
    'gauge': pd.DataFrame(index=dates, data={'discharge': range(10)}),
}


for basin in ['larry','curly','moe']:
    print(basin)
    for source, data in source_dict.items():
        print(source)
        
        data_dict = {}
        for subbasin in range(6):
            data_dict[f"{basin}_{subbasin}"] = data
    
        store.write_dynamic(basin, source, data_dict, mode='append')

In [ ]:
store.is_processed('larry','larry_0','ERA5')

In [ ]:
store.get_processing_status('larry','ERA5')

In [ ]:
df.xs('larry_0', level='subbasin')

In [ ]:
# 3. Read the data - it all works seamlessly
df = store.read_dynamic(basin='larry')#, concat_sources=False)
df

In [ ]:
df['source']

In [ ]:
store.compact()

In [ ]:
store.vacuum(retention_hours=0) 

In [ ]:
store.table_uri